# 07 · BM25 i hybrid search

Embeddingi dobrze łapią parafrazę, ale mogą przeoczyć dokładny kod, nazwę metryki lub identyfikator. BM25 ocenia pokrycie terminów z poprawką na częstość i długość dokumentu. Łączymy ranking BM25 i dense przez **Reciprocal Rank Fusion (RRF)**, a nie przez dodawanie nieporównywalnych surowych skal.

In [ ]:
# Jedyna komórka z importami i ścieżką projektu.
import math
import re
import sys
from pathlib import Path

import numpy as np
from rank_bm25 import BM25Okapi

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
sys.path.insert(0, str(ROOT / "notebooks" / "rag"))
from lab import embed_texts, section_records

## 1. Skąd bierze się wynik BM25?

Dla terminu `t` i dokumentu `d`: `IDF(t) × tf(t,d)(k1+1) / [tf(t,d)+k1(1-b+b·|d|/avgdl)]`. `tf` nasyca się: dziesiąte wystąpienie daje mniej niż pierwsze. `b` reguluje karę za długość, `k1` nasycenie. Popularny termin ma mniejsze IDF. Wersja `rank_bm25.BM25Okapi` stosuje własne wygładzenie IDF, więc poniższy rachunek jest intuicją, nie obietnicą identycznej liczby.

In [ ]:
toy = ["MASE uses a training denominator", "Percentage error can have zero denominator", "A forecast is frozen before the next month"]
tokens = [re.findall("[a-z0-9]+", text.lower()) for text in toy]
term = "denominator"
N = len(tokens)
df = sum(term in row for row in tokens)
idf = math.log(1 + (N - df + 0.5) / (df + 0.5))
{"df": df, "N": N, "illustrative_idf": round(idf, 3),
 "bm25_package_scores": BM25Okapi(tokens).get_scores([term]).round(3).tolist()}

Zobacz osobno dwa mechanizmy przy stałym `IDF`: powtórzenia terminu dają coraz mniejszy przyrost, a dłuższy fragment z tym samym jednym trafieniem otrzymuje niższą ocenę. To jest rachunek z podanego wzoru, więc wartości mogą różnić się od wygładzonej implementacji biblioteki.

In [ ]:
k1, b, average_length = 1.5, 0.75, 100
frequency_effect = [(tf, round(idf * tf * (k1 + 1) /
                               (tf + k1 * (1 - b + b * 100 / average_length)), 3))
                    for tf in (1, 2, 4, 8)]
length_effect = [(length, round(idf * (k1 + 1) /
                                (1 + k1 * (1 - b + b * length / average_length)), 3))
                 for length in (50, 100, 200)]
{"powtórzenia_tf": frequency_effect, "długość_przy_tf_1": length_effect}

## 2. Dwa rankingi tych samych sekcji

Tokenizację query i dokumentów wykonujemy tą samą regułą. Ten prosty tokenizer jest wystarczający dla angielskich dokumentów dydaktycznych, ale nie obsługuje odmiany polskiej, fraz ani specjalnych identyfikatorów. Model dense pracuje na oryginalnym tekście.

In [ ]:
records = section_records()
terms = [re.findall("[a-z0-9]+", row["text"].lower()) for row in records]
bm25 = BM25Okapi(terms)
matrix = embed_texts([row["text"] for row in records])
matrix /= np.linalg.norm(matrix, axis=1, keepdims=True)
question = "Why can MASE have an undefined training denominator?"
lexical = bm25.get_scores(re.findall("[a-z0-9]+", question.lower()))
q = embed_texts([question])[0]
q /= np.linalg.norm(q)
dense = matrix @ q
bm_rank = np.argsort(-lexical)[:8]
vec_rank = np.argsort(-dense)[:8]
{"BM25": [(records[int(i)]["id"], round(float(lexical[i]), 2)) for i in bm_rank[:4]],
 "dense": [(records[int(i)]["id"], round(float(dense[i]), 3)) for i in vec_rank[:4]]}

## 3. Reciprocal Rank Fusion

`RRF(d) = Σ 1/(k+rank(d))`; używamy rankingu, nie wartości BM25 ani cosinusa. `k=60` łagodzi wpływ jednego pierwszego miejsca. Kandydat spoza top-8 danego źródła nie wnosi punktów z tego źródła. Potem można przekazać top-k do rerankera.

In [ ]:
rrf = np.zeros(len(records))
for ranking in (bm_rank, vec_rank):
    for rank, index in enumerate(ranking, 1):
        rrf[index] += 1 / (60 + rank)
hybrid_rank = np.argsort(-rrf)[:6]
[{"id": records[int(i)]["id"], "heading": records[int(i)]["heading"],
  "rrf": round(float(rrf[i]), 4), "bm25": round(float(lexical[i]), 2),
  "cosine": round(float(dense[i]), 3)} for i in hybrid_rank]

## 4. Kiedy hybryda pomaga, a kiedy szkodzi?

BM25 pomaga przy `MASE`, kodach produktów i konkretnych liczbach. Dense pomaga przy pytaniach opisowych i parafrazach. Hybryda może pogorszyć wynik, gdy źle tokenizowane popularne słowo wypchnie trafną parafrazę. Filtr dostępu powinien ograniczyć **oba** zbiory kandydatów przed fuzją. Sprawdzaj recall@k na oznaczonych pytaniach; ranking nie jest prawdziwością.

Sprawdź dwa pytania opisowe: jedno o zerowy popyt, drugie o wpływ braku towaru na backtest. Porównaj pozycję właściwej sekcji w każdym rankingu. RRF może przesunąć ją w górę **lub** w dół; wyniki zależą od korpusu i modelu embeddingowego.

In [ ]:
cases = [
    ("How can we assess forecast accuracy when some periods have zero demand?", "SCI-FORECAST-2011:v1:s04"),
    ("How should stockout effects be handled in forecast backtests?", "SCI-STOCK-2011:v1:s04"),
]
case_vectors = embed_texts([text for text, _ in cases])
case_vectors /= np.linalg.norm(case_vectors, axis=1, keepdims=True)
comparisons = []
for (text, target_id), query_vector in zip(cases, case_vectors):
    lexical_scores = bm25.get_scores(re.findall("[a-z0-9]+", text.lower()))
    dense_scores = matrix @ query_vector
    bm_order, dense_order = np.argsort(-lexical_scores), np.argsort(-dense_scores)
    fused_scores = np.zeros(len(records))
    for ranking in (bm_order[:8], dense_order[:8]):
        for rank, index in enumerate(ranking, 1):
            fused_scores[index] += 1 / (60 + rank)
    target = next(i for i, row in enumerate(records) if row["id"] == target_id)
    comparisons.append({"query": text, "target": target_id,
        "rank_bm25": int(np.flatnonzero(bm_order == target)[0] + 1),
        "rank_dense": int(np.flatnonzero(dense_order == target)[0] + 1),
        "rank_hybrid": int(np.flatnonzero(np.argsort(-fused_scores) == target)[0] + 1)})
comparisons

Pozycja celu jest tylko jedną metryką. Dla większego zbioru pytań oznacz także wszystkie fragmenty wystarczające do odpowiedzi i porównaj recall@k lub nDCG@k. **Ćwiczenie:** zapytaj o dokładny identyfikator dokumentu i oceń, czy tokenizacja rozbija go poprawnie. [Biblioteka BM25](https://github.com/dorianbrown/rank_bm25) wyjaśnia również, że sama nie tokenizuje tekstu.